# Notebook 2: Data Cleaning

In [2]:
import pandas as pd
import numpy as np

print(" Starting Data Cleaning...")

# 1. Load Raw Data
df_spot = pd.read_csv("../data/nifty_spot_5min.csv")
df_fut = pd.read_csv("../data/nifty_futures_5min.csv")
df_opt = pd.read_csv("../data/nifty_options_5min.csv")

# 2. Robust Timestamp Alignment (Fixes 'KeyError: time')
def create_timestamp(df):
    df.columns = df.columns.str.lower() # Force lowercase
    if 'date' in df.columns and 'time' in df.columns:
        df['timestamp'] = pd.to_datetime(df['date'].astype(str) + ' ' + df['time'].astype(str))
    elif 'datetime' in df.columns:
        df['timestamp'] = pd.to_datetime(df['datetime'])
    df.set_index('timestamp', inplace=True)
    return df

df_spot = create_timestamp(df_spot)
df_fut = create_timestamp(df_fut)
df_opt = create_timestamp(df_opt)

# 3. Handling Outliers (Z-Score)
from scipy import stats
df_spot = df_spot[(np.abs(stats.zscore(df_spot['close'])) < 3)]

# 4. Merge Datasets
# Join Spot + Futures (suffix _fut) + Options (suffix _opt)
df_merged = df_spot.join(df_fut[['close', 'oi']], rsuffix='_fut')
df_merged = df_merged.join(df_opt[['atm_strike', 'call_iv', 'put_iv', 'call_oi', 'put_oi']], rsuffix='_opt')

# 5. Handle Missing Values
df_merged.fillna(method='ffill', inplace=True)
df_merged.dropna(inplace=True)

# 6. Save Cleaned Master File
df_merged.to_csv("../data/nifty_merged_5min.csv")
with open("../data/data_cleaning_report.txt", "w") as f:
    f.write(f"Data Cleaning Complete.\nFinal Shape: {df_merged.shape}\nMissing Values: 0")

print(f" Data Cleaned & Merged. Shape: {df_merged.shape}")

 Starting Data Cleaning...


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17192\435718888.py:35: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_merged.fillna(method='ffill', inplace=True)


 Data Cleaned & Merged. Shape: (27618, 15)
